Generates the independent-critic data for `paper plots/Trial shuffles.ipynb` (1 training iteration, no shuffle).


In [ ]:
import os, sys
cur_directory = os.getcwd()
project_folder = os.path.abspath(os.path.join(cur_directory, os.pardir, os.pardir))
save_directory = os.path.join(cur_directory, 'saved')
sys.path.append(os.path.join(project_folder, 'analysis-suite'))

In [ ]:
import numpy as np
from numpy import linalg as la
import matplotlib as mpl
from matplotlib import pyplot as plt
import analyses as als
import datasets as ds
import models as mds
import tools
from random import sample
from tqdm import tqdm
from argparse import Namespace
from time import gmtime, strftime
import pickle

In [ ]:
import numpy as np
import tensorflow as tf

tfkl = tf.keras.layers
import keras

In [ ]:
class AutocorrelationOnlyCritic(tf.keras.Model):
    
    def __init__(self, N_features, hidden_dim, embed_dim, layers, activation, dropout_rate=0):
        super(AutocorrelationOnlyCritic, self).__init__()

        self.sub_critics = []
        self.N_features = N_features
        for n in range(self.N_features):
            self.sub_critics.append(mds.SeparableCritic(hidden_dim, embed_dim, layers, activation, dropout_rate))
            
        self.f = tf.keras.Sequential([mds.mlp(10,10,3,'tanh'), tfkl.Dense(1)])
        self.g = tf.keras.Sequential([mds.mlp(10,10,3,'tanh'), tfkl.Dense(1)])
        self.pure = {'f':self.f, 'g':self.g} # for easy testing

    @tf.function
    def call(self, x, y):
        out = tfkl.add([self.sub_critics[n](x[:,n][:,None], y[:,n][:,None]) for n in range(self.N_features)]) + tf.transpose(self.f(x)) + self.g(y)
        # out = tfkl.add([self.sub_critics[n](x[:,n][:,None], y[:,n][:,None]) for n in range(self.N_features)])
        return out

In [ ]:
data_path = os.path.join(project_folder, 'data', 'nat_scenes_data')
D = ds.DataNatScenes(data_path, mov_ind=4)
print(f'current stimulus is {D.movname}')
mov_ind = D.mov_ind

In [ ]:
V_width = 1
V_E_buffer = 0
E_width = 1
N_trials = 10

opt_params = {"track_quantities": False, "quiet": True, "iterations": 1}
mi_measurement_params = {"quiet": True, "batch_size": 500, "N_batches": 500}

critic_params_ctl = {
    "hidden_dim": 32,
    "embed_dim": 30,
    "layers": 2,
    "activation": "tanh",
}

critic_params = {
    "N_features": D.N * V_width,
    "hidden_dim": 3,
    "embed_dim": 3,
    "layers": 2,
    "activation": "tanh",
}

shuffle_trial_per_time = False

# critic_params = {
#     "N_features": D.N * V_width,
#     "layers": 1,
#     "activation": "tanh",
# }

experiment_configuration_variables = {'mi_measurement_params', 'N_trials', 'opt_params', 'V_width', 'V_E_buffer', 'E_width', 'critic_params', 'critic_params_ctl', 'shuffle_trial_per_time'}

In [ ]:
D.reset()
D.trim(20)
# D.subset_features(range(10))
if shuffle_trial_per_time:
    D.shuffle_trial_per_time()
V, E = D.create_V_and_E_datasets(V_width,V_E_buffer,E_width,1)

## Run test

In [ ]:
def test_model(critic_function):
    V.reset()
    E.reset()
    train_points, test_points = tools.generate_train_test_indices(V.P, 0.5)

    # training set
    V.subset_points(train_points)
    E.subset_points(train_points)
    
    als.train_NCE(
        X=V.X,
        Y=E.X,
        critic_function=critic_function,
        **opt_params
    )

    # training set
    mi_train = als.MI_NCE(
        X=V.X, Y=E.X, critic_function=critic_function, **mi_measurement_params
    )

    # Measure InfoNCE for test
    V.reset()
    E.reset()
    V.subset_points(test_points)
    E.subset_points(test_points)
    mi_test = als.MI_NCE(
        X=V.X, Y=E.X, critic_function=critic_function, **mi_measurement_params
    )

    return mi_train, mi_test

In [ ]:
# critic_weights = []

experiment_dims = (N_trials,)
mi_averages_test = np.zeros(experiment_dims)
mi_averages_train = np.zeros(experiment_dims)
mi_stdevs_train = np.zeros(experiment_dims)
mi_stdevs_test = np.zeros(experiment_dims)

mi_averages_test_ctl = np.zeros(experiment_dims)
mi_averages_train_ctl = np.zeros(experiment_dims)
mi_stdevs_train_ctl = np.zeros(experiment_dims)
mi_stdevs_test_ctl = np.zeros(experiment_dims)

for n in range(N_trials):
    critic_function_ctl = mds.SeparableCritic(**critic_params_ctl)
    critic_function = AutocorrelationOnlyCritic(**critic_params)
    mi_train, mi_test = test_model(critic_function)
    mi_train_ctl, mi_test_ctl = test_model(critic_function_ctl)
    mi_averages_test[n] = mi_test["mi"]
    mi_stdevs_train[n] = mi_train["mi_stdev"]
    mi_averages_train[n] = mi_train["mi"]
    mi_stdevs_test[n] = mi_test["mi_stdev"]
    mi_averages_test_ctl[n] = mi_test_ctl["mi"]
    mi_stdevs_test_ctl[n] = mi_test_ctl["mi_stdev"]
    mi_averages_train_ctl[n] = mi_train_ctl["mi"]
    mi_stdevs_train_ctl[n] = mi_train_ctl["mi_stdev"]
    

## Save

In [ ]:
# select variables
critic_type = type(critic_function)
critic_config = critic_function.get_config()
save_vars = {
    "mi_averages_train",
    "mi_averages_test",
    "mi_stdevs_train",
    "mi_stdevs_test",
    "mi_averages_train_ctl",
    "mi_averages_test_ctl",
    "mi_stdevs_train_ctl",
    "mi_stdevs_test_ctl",
    "critic_type",
    "critic_config"
}.union(experiment_configuration_variables)
local_dict = locals()
output_data = {k: local_dict[k] for k in save_vars}

In [ ]:
# save things
timestamp = strftime("%Y.%m.%d %Hh %Mm %Ss.dat",gmtime())
# full_file_path = os.path.join(save_directory, '', strftime("%Y.%m.%d %Hh %Mm %Ss",gmtime())+'.dat')

stim = D.movname
save_directory = os.path.join(cur_directory, 'saved', stim)
file_name = f'autocorrelations only {(V_width,V_E_buffer,E_width)}, hidden_dim:{critic_config["hidden_dim"]}, embed_dim:{critic_config["embed_dim"]}, layers:{critic_config["layers"]}'
if opt_params["iterations"] != 1:
    file_name += f', iter:{opt_params["iterations"]}'
if shuffle_trial_per_time:
    file_name += ', shuffle_trial_per_time:True,'
full_file_path = os.path.join(save_directory, file_name + ' with pure.dat')
# full_file_path = os.path.join(save_directory, 'test.dat')
# print(f"will save to {save_directory}")
print(f"will save to {full_file_path}")

In [ ]:
os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
with open(full_file_path, 'wb+') as f:
            pickle.dump(output_data, f)
tools.record_parameters(full_file_path, output_data)